# Geleneksel Derin Öğrenme Modelleri (Bi-LSTM ve TextCNN)

Bu notebook'ta, BERT gibi hazır (pre-trained) Transformer modelleri kullanmak yerine, tamamen sıfırdan kendi derin öğrenme ağımızı kuracağız.

Kendi kelime sözlüğümüzü (Vocabulary) oluşturacak ve metinleri PyTorch kullanarak iki farklı klasik NLP mimarisi ile eğiteceğiz:
1. **Bi-LSTM (Bidirectional Long Short-Term Memory):** Kelimeleri dizisel (zaman serisi) olarak hem ileri hem geri okuyan efsanevi yapı.
2. **TextCNN (Convolutional Neural Network):** Metin üzerinde 1-boyutlu evrişim (Convolution) yaparak n-gram (yan yana kelime) özelliklerini yakalayan hızlı yapı.

Bu modeller Transformer'lara göre çok daha hızlı eğitilir ve kendi ağırlıklarını sıfırdan öğrendikleri için projenize büyük prestij katar.

## Bölüm 1 — Ortam Kurulumu ve Veri Yükleme

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from torch.utils.data import Dataset, DataLoader
from collections import Counter

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Cihaz: {device}")

# Veri Yükleme
df = pd.read_csv('data/processed/reviews_cleaned.csv').dropna(subset=['cleaned_text', 'label'])
min_class_size = df['label'].value_counts().min()
df_balanced = df.groupby('label').apply(lambda x: x.sample(min_class_size, random_state=SEED)).reset_index(drop=True)

label2id = {'negative': 0, 'neutral': 1, 'positive': 2}
df_balanced['label'] = df_balanced['label'].map(label2id)

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df_balanced['cleaned_text'].tolist(), df_balanced['label'].tolist(), test_size=0.2, random_state=SEED, stratify=df_balanced['label'].tolist()
)


## Bölüm 2 — Sözlük (Vocabulary) Oluşturma ve Tokenizasyon

In [ ]:
# Basit Tokenizer (Boşluklara göre ayırma)
def tokenize(text):
    return str(text).lower().split()

# Kelime Frekansları
word_counts = Counter()
for text in train_texts:
    word_counts.update(tokenize(text))

# En sık geçen 20,000 kelimeyi sözlüğe al (Gerisi <UNK> olacak)
MAX_VOCAB_SIZE = 20000
vocab = {word: i + 2 for i, (word, _) in enumerate(word_counts.most_common(MAX_VOCAB_SIZE))}
vocab['<PAD>'] = 0
vocab['<UNK>'] = 1

def encode_text(text, max_len=128):
    tokens = tokenize(text)
    encoded = [vocab.get(t, vocab['<UNK>']) for t in tokens][:max_len]
    padding = [vocab['<PAD>']] * (max_len - len(encoded))
    return encoded + padding

train_encodings = [encode_text(t) for t in train_texts]
test_encodings = [encode_text(t) for t in test_texts]

class DLDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = torch.tensor(encodings, dtype=torch.long)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.encodings[idx], self.labels[idx]

train_loader = DataLoader(DLDataset(train_encodings, train_labels), batch_size=64, shuffle=True)
test_loader = DataLoader(DLDataset(test_encodings, test_labels), batch_size=64)


## Bölüm 3 — Bi-LSTM Mimarisi

In [ ]:
class BiLSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim, n_layers=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=n_layers, bidirectional=True, dropout=dropout, batch_first=True)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, text):
        embedded = self.dropout(self.embedding(text))
        _, (hidden, _) = self.lstm(embedded)
        # hidden_state: [num_layers * num_directions, batch, hidden_dim]
        # Son katmanın ileri (forward) ve geri (backward) çıktılarını birleştir
        hidden = self.dropout(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1))
        return self.fc(hidden)

bilstm_model = BiLSTMModel(len(vocab), embed_dim=128, hidden_dim=128, output_dim=3).to(device)
print(bilstm_model)


## Bölüm 4 — TextCNN Mimarisi

In [ ]:
import torch.nn.functional as F

class TextCNNModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, n_filters, filter_sizes, output_dim, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=embed_dim, out_channels=n_filters, kernel_size=fs)
            for fs in filter_sizes
        ])
        self.fc = nn.Linear(len(filter_sizes) * n_filters, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, text):
        embedded = self.embedding(text) # [batch size, sent len, emb dim]
        embedded = embedded.permute(0, 2, 1) # [batch size, emb dim, sent len]
        
        conved = [F.relu(conv(embedded)) for conv in self.convs]
        pooled = [F.max_pool1d(conv, conv.shape[2]).squeeze(2) for conv in conved]
        
        cat = self.dropout(torch.cat(pooled, dim=1))
        return self.fc(cat)

textcnn_model = TextCNNModel(len(vocab), embed_dim=128, n_filters=100, filter_sizes=[3,4,5], output_dim=3).to(device)
print(textcnn_model)


## Bölüm 5 — Eğitim Döngüsü (Training Loop)

In [ ]:
def train_model(model, train_loader, test_loader, epochs=5, lr=0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch_texts, batch_labels in train_loader:
            batch_texts, batch_labels = batch_texts.to(device), batch_labels.to(device)
            optimizer.zero_grad()
            predictions = model(batch_texts)
            loss = criterion(predictions, batch_labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        # Evaluation
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for batch_texts, batch_labels in test_loader:
                batch_texts = batch_texts.to(device)
                predictions = model(batch_texts)
                preds = torch.argmax(predictions, dim=1).cpu().numpy()
                all_preds.extend(preds)
                all_labels.extend(batch_labels.numpy())

        acc = accuracy_score(all_labels, all_preds)
        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f} | Test Acc: {acc:.4f}")

# Bi-LSTM Eğitimi (Çalıştırmak için yorum satırını kaldırın)
print("--- Bi-LSTM Eğitimi ---")
train_model(bilstm_model, train_loader, test_loader, epochs=5)
torch.save(bilstm_model.state_dict(), './models/bilstm_model.pt')

# TextCNN Eğitimi (Çalıştırmak için yorum satırını kaldırın)
print("\n--- TextCNN Eğitimi ---")
train_model(textcnn_model, train_loader, test_loader, epochs=5)
torch.save(textcnn_model.state_dict(), './models/textcnn_model.pt')


In [ ]:
import json
# Sözlüğü de kaydetmemiz lazım (Inference için)
with open('./models/dl_vocab.json', 'w') as f:
    json.dump(vocab, f)
